#  Exploration & Nettoyage
Source : CNSR via Open Data Bénin (2010–2022)

> *Au Bénin, les accidents baissent. Les morts, eux, augmentent.*

## 0. Imports

In [10]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

YEARS = list(range(2010, 2023))
DEPTS = ['Alibori','Atakora','Atlantique','Borgou','Collines',
         'Kouffo','Donga','Littoral','Mono','Ouémé','Plateau','Zou']

## 1. Chargement & Nettoyage — Dataset Gravité (national)

In [11]:
df_raw = pd.read_excel('../data/raw/ObservationData_gravite.xlsx', engine='calamine')

df_raw.columns = [
    'annee',
    'acc_materiels','acc_corp_legers','acc_graves_nm','acc_mortels',
    'veh_materiels','veh_corp_legers','veh_graves_nm','veh_mortels',
    'tues_materiels','tues_corp_legers','tues_graves_nm','tues_mortels',
    'bg_materiels','bg_corp_legers','bg_graves_nm','bg_mortels',
    'bl_materiels','bl_corp_legers','bl_graves_nm','bl_mortels'
]

df_gravite = df_raw[
    df_raw['annee'].apply(lambda x: str(x).isdigit() if pd.notna(x) else False)
].copy()

df_gravite = df_gravite.apply(pd.to_numeric, errors='coerce')
df_gravite = df_gravite[df_gravite['annee'] >= 2010].reset_index(drop=True)

# Total accidents et tués (toutes gravités)
df_gravite['total_accidents'] = df_gravite[['acc_materiels','acc_corp_legers','acc_graves_nm','acc_mortels']].sum(axis=1)
df_gravite['total_tues'] = df_gravite[['tues_mortels']].sum(axis=1)  # seule catégorie avec des tués
df_gravite['taux_mortalite'] = (df_gravite['total_tues'] / df_gravite['total_accidents'] * 1000).round(1)

print('Shape:', df_gravite.shape)
df_gravite[['annee','total_accidents','acc_mortels','total_tues','taux_mortalite']]

Shape: (13, 24)


,annee,total_accidents,acc_mortels,total_tues,taux_mortalite
0,2010,4863,601,760,156.3
1,2011,5986,638,791,132.1
2,2012,5740,534,658,114.6
3,2013,107,45,55,514.0
4,2014,6169,550,670,108.6
5,2015,80,26,39,487.5
6,2016,5946,545,629,105.8
7,2017,5717,571,676,118.2
8,2018,5386,617,740,137.4
9,2019,5345,688,817,152.9


## 2. Chargement & Nettoyage — Dataset Départemental

In [12]:
df_raw2 = pd.read_excel('../data/raw/Accidents_victimes_par_Departement.xlsx', engine='calamine', skiprows=5)
df_raw2.columns = ['Indicateur', 'Zone', 'Units'] + YEARS

# Filtrer les 12 départements uniquement
df_dept_wide = df_raw2[
    df_raw2['Zone'].isin(DEPTS) &
    df_raw2['Indicateur'].isin(['Accidents','Personnes tués','Blessés graves','Blessés légers','Véhicules impliqués'])
].drop(columns=['Units']).copy()

# Passer en format long
df_dept_long = df_dept_wide.melt(
    id_vars=['Indicateur','Zone'],
    value_vars=YEARS,
    var_name='annee',
    value_name='valeur'
)
df_dept_long['annee'] = df_dept_long['annee'].astype(int)
df_dept_long['valeur'] = pd.to_numeric(df_dept_long['valeur'], errors='coerce')

# Pivoter pour avoir une ligne par (département, année)
df_dept = df_dept_long.pivot_table(
    index=['Zone','annee'],
    columns='Indicateur',
    values='valeur'
).reset_index()
df_dept.columns.name = None
df_dept.rename(columns={
    'Zone': 'departement',
    'Accidents': 'accidents',
    'Personnes tués': 'tues',
    'Blessés graves': 'blesses_graves',
    'Blessés légers': 'blesses_legers',
    'Véhicules impliqués': 'vehicules'
}, inplace=True)

df_dept['taux_mortalite'] = (df_dept['tues'] / df_dept['accidents'] * 1000).round(1)

print('Shape:', df_dept.shape)
df_dept.head(12)

Shape: (156, 8)


,departement,annee,accidents,blesses_graves,blesses_legers,tues,vehicules,taux_mortalite
0,Alibori,2010,137.0,60.0,93.0,45.0,233.0,328.5
1,Alibori,2011,128.0,106.0,101.0,71.0,222.0,554.7
2,Alibori,2012,131.0,74.0,81.0,51.0,233.0,389.3
3,Alibori,2013,107.0,61.0,55.0,55.0,186.0,514.0
4,Alibori,2014,90.0,43.0,36.0,35.0,172.0,388.9
5,Alibori,2015,80.0,79.0,53.0,39.0,138.0,487.5
6,Alibori,2016,104.0,49.0,39.0,28.0,178.0,269.2
7,Alibori,2017,438.0,138.0,170.0,67.0,826.0,153.0
8,Alibori,2018,221.0,86.0,139.0,54.0,363.0,244.3
9,Alibori,2019,215.0,93.0,145.0,83.0,371.0,386.0


## 3. Insight #1 — Le paradoxe : moins d'accidents, plus de morts

In [13]:
# Données nationales depuis le dataset départemental (ligne Bénin)
df_raw2_benin = df_raw2[df_raw2['Zone']=='Bénin'].copy()
benin = {}
for ind in ['Accidents','Personnes tués','Blessés graves']:
    row = df_raw2_benin[df_raw2_benin['Indicateur']==ind].iloc[0]
    benin[ind] = [row[y] for y in YEARS]

df_benin = pd.DataFrame({'annee': YEARS,
                         'accidents': benin['Accidents'],
                         'tues': benin['Personnes tués'],
                         'blesses_graves': benin['Blessés graves']})
df_benin['taux_mortalite'] = (df_benin['tues'] / df_benin['accidents'] * 1000).round(1)

# Graphique double axe
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=df_benin['annee'], y=df_benin['accidents'],
    name='Accidents', marker_color='#94a3b8', opacity=0.7
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_benin['annee'], y=df_benin['tues'],
    name='Personnes tuées', mode='lines+markers',
    line=dict(color='#ef4444', width=3),
    marker=dict(size=8)
), secondary_y=True)

fig.update_layout(
    title='Bénin 2010–2022 : Accidents vs Personnes tuées',
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.update_yaxes(title_text='Nombre d\'accidents', secondary_y=False)
fig.update_yaxes(title_text='Personnes tuées', secondary_y=True)
fig.show()

print('\nTaux de mortalité (tués/1000 accidents):')
print(df_benin[['annee','taux_mortalite']].to_string(index=False))


Taux de mortalité (tués/1000 accidents):
 annee  taux_mortalite
  2010           156.3
  2011           132.1
  2012           114.6
  2013           125.7
  2014           108.6
  2015           106.3
  2016           105.9
  2017           117.3
  2018           137.4
  2019           152.9
  2020           198.8
  2021           218.7
  2022           250.1



Accidents vs Personnes tuées (national)
 Les accidents oscillent entre 4500 et 6000 sur toute la période pas de tendance claire. Mais les tués, eux, dessinent une courbe en J parfaite : relativement stable de 2010 à 2018 autour de 650–800, puis explosion verticale à partir de 2019. En 2022 on atteint 1224 morts pour seulement 4894 accidents  le niveau le plus bas d'accidents de toute la période. C'est le paradoxe central du dashboard.
Ce qui se passe probablement : pas plus d'accidents, mais des accidents beaucoup plus graves. Vitesse excessive, 2-roues sans casque, infrastructure qui ne suit pas la croissance du parc automobile.

## 4. Insight #2 — Taux de mortalité par département (2022)

In [14]:
df_2022 = df_dept[df_dept['annee']==2022].sort_values('taux_mortalite', ascending=True)

fig2 = px.bar(
    df_2022,
    x='taux_mortalite', y='departement',
    orientation='h',
    color='taux_mortalite',
    color_continuous_scale='Reds',
    title='Taux de mortalité par département — 2022 (tués pour 1000 accidents)',
    labels={'taux_mortalite': 'Tués / 1000 accidents', 'departement': ''}
)
fig2.update_layout(plot_bgcolor='white', showlegend=False)
fig2.show()

Taux de mortalité par département 2022
Zou est en tête avec ~340 tués pour 1000 accidents  c'est colossal. Puis Alibori, Kouffo, Atakora dans les 260–270. Plateau et Collines ferment le classement sous 220. Ce qui est notable : Littoral (Cotonou) est dans la moyenne basse malgré la densité de trafic. Les départements ruraux tuent plus par accident  routes en mauvais état, éloignement des centres de soins, délais d'intervention.

## 5. Insight #3 — Évolution du taux de mortalité par département

In [15]:
fig3 = px.line(
    df_dept,
    x='annee', y='taux_mortalite',
    color='departement',
    title='Évolution du taux de mortalité par département (2010–2022)',
    labels={'taux_mortalite': 'Tués / 1000 accidents', 'annee': 'Année'}
)
fig3.update_layout(plot_bgcolor='white')
fig3.show()

 Évolution par département 2010–2022
Deux observations majeures :
2016–2017 : chute brutale de tous les départements vers 100–150. Probablement un changement de méthode de collecte ou une réforme CNSR à noter dans le dashboard comme biais possible.
Post-2018 : remontée générale et divergence. Certains départements comme Zou et Alibori s'envolent vers 400+, d'autres comme Atlantique et Littoral restent bas. La disparité géographique se creuse sur les dernières années.

## 6. Export données nettoyées

In [16]:
df_dept.to_csv('../data/processed/accidents_departements.csv', index=False)
df_benin.to_csv('../data/processed/accidents_national.csv', index=False)
df_gravite.to_csv('../data/processed/gravite_nationale.csv', index=False)
print('✅ Exports terminés')
print('  → data/processed/accidents_departements.csv', df_dept.shape)
print('  → data/processed/accidents_national.csv', df_benin.shape)
print('  → data/processed/gravite_nationale.csv', df_gravite.shape)

✅ Exports terminés
  → data/processed/accidents_departements.csv (156, 8)
  → data/processed/accidents_national.csv (13, 5)
  → data/processed/gravite_nationale.csv (13, 24)
